In [0]:
--========================================================================
--DATA EXPLORATION
--1. Initial Data Inspection
--========================================================================
SELECT * 
FROM `workspace`.`default`.`car_sales_data_`
LIMIT 10;
--===========================================================================
-- 2. Checking total sales count
--===========================================================================
SELECT COUNT(*) AS total_sales
FROM `workspace`.`default`.`car_sales_data_`;

--Total sales = 558811

--============================================================================
--3. Checking unique cars sold
--============================================================================
SELECT 
  COUNT(DISTINCT vin) AS unique_vehicles
FROM `workspace`.`default`.`car_sales_data_`;
 
 --We have 550296 unique cars sold

--=============================================================================
-- 4. Checking for missing values in key columns
--=============================================================================
SELECT 
    COUNT(*) - COUNT(make) AS missing_make,          -- 10301 Missing Values
    COUNT(*) - COUNT(model) AS missing_model,        ----10399 Missing Values
    COUNT(*) - COUNT(sellingprice) AS missing_price,  -- 12   Missing Values
    COUNT(*) - COUNT(saledate) AS missing_date        --12   Missing Values
FROM `workspace`.`default`.`car_sales_data_`;

--==============================================================================
-- 5. Identify unique car makes and count of entries per make
--==============================================================================
SELECT make, COUNT(*) as listings
FROM `workspace`.`default`.`car_sales_data_`
GROUP BY make
ORDER BY listings DESC;

-- We have 97 car makes with Ford having the highest listing, followed by Chevrolet & Nissan

--===============================================================================
-- 5. Identify the date range of the sales
--===============================================================================
SELECT MIN(saledate) AS Sales_start_date,
       MAX(saledate) AS Sales_end_date
FROM `workspace`.`default`.`car_sales_data_`;

--Sales starts from 03 April 2015 and ends 27 May 2015

--===============================================================================
--Creating a processed table with calculated metrics
--=============================================================================
SELECT 
    vin,

    -- Clean date
    to_timestamp(substring(saledate, 5), 'MMM dd yyyy HH:mm:ss') AS sale_date,

    -- Clean text fields
    trim(make) AS make,
    trim(model) AS model,

    -- New column (car name)
    concat(trim(make), ' ', trim(model)) AS car_name,

    -- Other columns
    cast(year AS INT) AS manufacture_year,
    trim(state) AS region,
    trim(body) AS body_type,

    -- Numeric conversions
    cast(sellingprice AS DOUBLE) AS selling_price,
    cast(mmr AS DOUBLE) AS cost_price,
    cast(odometer AS DOUBLE) AS mileage,

    -- Calculations
    cast(sellingprice AS DOUBLE) AS total_revenue,
    (sellingprice - mmr) AS total_profit,
    ROUND(((sellingprice - mmr) / NULLIF(sellingprice, 0)) * 100, 2) AS profit_margin_percentage,

    -- Date breakdowns
    year(to_timestamp(substring(saledate, 5), 'MMM dd yyyy HH:mm:ss')) AS sale_year,
    month(to_timestamp(substring(saledate, 5), 'MMM dd yyyy HH:mm:ss')) AS sale_month_number,
    date_format(to_timestamp(substring(saledate, 5), 'MMM dd yyyy HH:mm:ss'), 'MMMM') AS sale_month_name,

CASE 
    WHEN ((sellingprice - mmr) / NULLIF(sellingprice, 0)) * 100 >= 10 THEN 'High'
    WHEN ((sellingprice - mmr) / NULLIF(sellingprice, 0)) * 100 >= 0 THEN 'Medium'
    ELSE 'Low'
END AS performance_tier

FROM `workspace`.`default`.`car_sales_data_`

WHERE sellingprice IS NOT NULL
  AND mmr IS NOT NULL
  AND saledate IS NOT NULL;